In [1]:
"""
NFL fair-odds model (ESPN stats) + Sleeper injuries + Elastic-Net Logistic Regression (BEST logistic upgrade)

Install:
  pip install requests pandas numpy scikit-learn

What this does:
1) Pull completed games for last N days from ESPN scoreboard.
2) Pull ESPN summary/boxscore for each completed game -> extract TEAM stats.
3) Build rolling pre-game features per team, then HOME-AWAY diffs.
4) Train Elastic-Net Logistic Regression + calibrate probabilities.
5) Predict upcoming games, then apply injury log-odds adjustment using Sleeper injury tags.
6) Print:
   - test accuracy/logloss
   - latest completed week odds
   - upcoming odds (base vs injury-adjusted)
   - upcoming injury list (who is injured)

Notes:
- ESPN injury endpoint is unreliable; Sleeper is better for injury_status.
- Injury adjustment is conservative & can be tuned later.
"""

from __future__ import annotations

import time
import math
import os
import json
import requests
import numpy as np
import pandas as pd

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Dict, Any, List, Optional, Tuple

from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression


# -----------------------------
# Config
# -----------------------------
ESPN_BASE = "https://site.web.api.espn.com/apis/site/v2/sports/football/nfl"

DEFAULT_N_DAYS_HISTORY = 240
ROLL_WINDOW = 5
SLEEP_BETWEEN_CALLS = 0.35

MIN_NON_NULL_RATE = 0.25
DROP_CONSTANT_FEATURES = True

# Elastic-net search space
ELASTIC_C_VALUES = [0.1, 0.3, 0.7, 1.5, 3.0]        # regularization strength
ELASTIC_L1_RATIOS = [0.05, 0.15, 0.30, 0.50, 0.70]  # mix between L1/L2

# Injuries (Sleeper)
USE_INJURY_ADJUSTMENT = True
PRINT_UPCOMING_INJURY_TABLE = True

SLEEPER_PLAYERS_URL = "https://api.sleeper.app/v1/players/nfl"
SLEEPER_CACHE_FILE = "sleeper_players_nfl_cache.json"
SLEEPER_CACHE_MAX_AGE_HOURS = 6

# Injury log-odds penalties (conservative)
INJURY_LOGIT_WEIGHTS = {
    "qb_out": 0.85,
    "qb_doubtful": 0.55,
    "qb_questionable": 0.20,

    "out_total": 0.015,
    "doubtful_total": 0.010,
    "questionable_total": 0.004,
    "ir_total": 0.002,
}


# -----------------------------
# Odds helpers
# -----------------------------
def to_american_odds(p: float) -> float:
    p = float(p)
    p = min(max(p, 1e-6), 1 - 1e-6)
    if p >= 0.5:
        return -100.0 * p / (1.0 - p)
    return 100.0 * (1.0 - p) / p


def safe_float(x: Any) -> float:
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        s = x.strip().replace("%", "")
        if s == "":
            return np.nan
        if any(ch in s for ch in ["-", "/"]) and not s.replace("-", "").replace("/", "").replace(".", "").isdigit():
            return np.nan
        try:
            return float(s)
        except Exception:
            return np.nan
    if isinstance(x, dict):
        for k in ("value", "displayValue", "stat", "amount"):
            if k in x:
                return safe_float(x[k])
        return np.nan
    return np.nan


def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))


def logit(p: float) -> float:
    p = float(p)
    p = min(max(p, 1e-6), 1 - 1e-6)
    return math.log(p / (1.0 - p))


# -----------------------------
# Sleeper injuries (reliable)
# -----------------------------
def _cache_is_fresh(path: str, max_age_hours: int) -> bool:
    if not os.path.exists(path):
        return False
    age_seconds = time.time() - os.path.getmtime(path)
    return age_seconds <= max_age_hours * 3600


def load_sleeper_players(force_refresh: bool = False) -> Dict[str, Any]:
    if (not force_refresh) and _cache_is_fresh(SLEEPER_CACHE_FILE, SLEEPER_CACHE_MAX_AGE_HOURS):
        try:
            with open(SLEEPER_CACHE_FILE, "r") as f:
                return json.load(f)
        except Exception:
            pass

    r = requests.get(SLEEPER_PLAYERS_URL, timeout=90)
    r.raise_for_status()
    data = r.json()

    try:
        with open(SLEEPER_CACHE_FILE, "w") as f:
            json.dump(data, f)
    except Exception:
        pass

    return data


def normalize_sleeper_injury(status: Optional[str]) -> str:
    if not status:
        return "healthy"
    s = str(status).strip().lower()
    if "out" in s:
        return "out"
    if "doubt" in s:
        return "doubtful"
    if "question" in s:
        return "questionable"
    if s == "ir" or "injured reserve" in s:
        return "ir"
    if "pup" in s:
        return "ir"
    return "other"


def summarize_team_injuries_from_sleeper(players_db: Dict[str, Any], team_abbr: str) -> Tuple[Dict[str, float], pd.DataFrame]:
    team_abbr = str(team_abbr).upper().strip()

    feats = {
        "inj_out_total": 0.0,
        "inj_doubtful_total": 0.0,
        "inj_questionable_total": 0.0,
        "inj_ir_total": 0.0,
        "inj_qb_out": 0.0,
        "inj_qb_doubtful": 0.0,
        "inj_qb_questionable": 0.0,
    }

    rows = []
    for _, p in players_db.items():
        if not isinstance(p, dict):
            continue
        if str(p.get("team") or "").upper() != team_abbr:
            continue

        injury_status = p.get("injury_status")
        bucket = normalize_sleeper_injury(injury_status)

        if bucket == "out":
            feats["inj_out_total"] += 1
        elif bucket == "doubtful":
            feats["inj_doubtful_total"] += 1
        elif bucket == "questionable":
            feats["inj_questionable_total"] += 1
        elif bucket == "ir":
            feats["inj_ir_total"] += 1

        pos = p.get("position")
        if str(pos).upper() == "QB":
            if bucket == "out":
                feats["inj_qb_out"] += 1
            elif bucket == "doubtful":
                feats["inj_qb_doubtful"] += 1
            elif bucket == "questionable":
                feats["inj_qb_questionable"] += 1

        if bucket in {"out", "doubtful", "questionable", "ir"}:
            rows.append({
                "team": team_abbr,
                "player": p.get("full_name") or ((p.get("first_name") or "") + " " + (p.get("last_name") or "")).strip(),
                "pos": p.get("position"),
                "injury_status": injury_status,
                "bucket": bucket,
            })

    inj_df = pd.DataFrame(rows)
    if not inj_df.empty:
        order = {"out": 0, "doubtful": 1, "questionable": 2, "ir": 3}
        inj_df["__ord"] = inj_df["bucket"].map(order).fillna(99)
        inj_df = inj_df.sort_values(["__ord", "pos", "player"]).drop(columns="__ord").reset_index(drop=True)

    return feats, inj_df


def injury_logit_penalty(team_inj_feats: Dict[str, float]) -> float:
    w = INJURY_LOGIT_WEIGHTS
    penalty = 0.0

    penalty += w["out_total"] * team_inj_feats.get("inj_out_total", 0.0)
    penalty += w["doubtful_total"] * team_inj_feats.get("inj_doubtful_total", 0.0)
    penalty += w["questionable_total"] * team_inj_feats.get("inj_questionable_total", 0.0)
    penalty += w["ir_total"] * team_inj_feats.get("inj_ir_total", 0.0)

    penalty += w["qb_out"] * team_inj_feats.get("inj_qb_out", 0.0)
    penalty += w["qb_doubtful"] * team_inj_feats.get("inj_qb_doubtful", 0.0)
    penalty += w["qb_questionable"] * team_inj_feats.get("inj_qb_questionable", 0.0)

    return float(penalty)


# -----------------------------
# ESPN fetching
# -----------------------------
@dataclass
class ESPNClient:
    sleep: float = SLEEP_BETWEEN_CALLS

    def __post_init__(self):
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"})

    def get_scoreboard(self, date_yyyymmdd: Optional[str] = None) -> Optional[Dict[str, Any]]:
        url = f"{ESPN_BASE}/scoreboard"
        if date_yyyymmdd:
            url += f"?dates={date_yyyymmdd}"
        try:
            r = self.session.get(url, timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[scoreboard] error for {date_yyyymmdd}: {e}")
            return None

    def get_summary(self, event_id: str) -> Optional[Dict[str, Any]]:
        url = f"{ESPN_BASE}/summary?event={event_id}"
        try:
            r = self.session.get(url, timeout=25)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[summary] error for event={event_id}: {e}")
            return None


def daterange_yyyymmdd(days_back: int) -> List[str]:
    today_utc = datetime.now(timezone.utc).date()
    return [(today_utc - timedelta(days=i)).strftime("%Y%m%d") for i in range(days_back)]


def collect_events_by_dates(client: ESPNClient, days_back: int) -> List[Dict[str, Any]]:
    dates = daterange_yyyymmdd(days_back)
    events: List[Dict[str, Any]] = []

    for ds in dates:
        data = client.get_scoreboard(ds)
        if data and "events" in data:
            events.extend(data["events"])
        time.sleep(client.sleep)

    seen = set()
    uniq = []
    for e in events:
        eid = str(e.get("id"))
        if not eid or eid in seen:
            continue
        seen.add(eid)
        uniq.append(e)
    return uniq


def parse_scoreboard_event_minimal(event_obj: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    try:
        eid = str(event_obj["id"])
        date_str = event_obj.get("date")
        game_dt = pd.to_datetime(date_str).tz_convert(None) if date_str else pd.NaT

        competition = event_obj["competitions"][0]
        status = competition["status"]["type"]
        completed = bool(status.get("completed", False))

        competitors = competition["competitors"]
        if len(competitors) != 2:
            return None

        teams = []
        for c in competitors:
            team = c["team"]
            teams.append({
                "event_id": eid,
                "game_date": game_dt,
                "team_id": str(team.get("id")),
                "team": team.get("displayName"),
                "abbr": team.get("abbreviation"),
                "home_away": c.get("homeAway"),
                "score": safe_float(c.get("score")),
                "winner": bool(c.get("winner", False)),
                "completed": completed,
            })
        return {"event_id": eid, "game_date": game_dt, "completed": completed, "teams": teams}
    except Exception:
        return None


def extract_team_stats_from_summary(summary_json: Dict[str, Any]) -> Dict[str, Dict[str, float]]:
    out: Dict[str, Dict[str, float]] = {}
    box = summary_json.get("boxscore", {}) if isinstance(summary_json, dict) else {}
    teams = box.get("teams", [])
    if not isinstance(teams, list):
        return out

    for t in teams:
        try:
            team_info = t.get("team", {})
            team_id = str(team_info.get("id"))
            if not team_id:
                continue

            stats_map: Dict[str, float] = {}
            stats_list = t.get("statistics", [])
            if isinstance(stats_list, list):
                for s in stats_list:
                    name = s.get("name") or s.get("abbreviation")
                    if not name:
                        continue
                    val = s.get("value")
                    if val is None:
                        val = s.get("displayValue")
                    stats_map[name] = safe_float(val)

            out[team_id] = stats_map
        except Exception:
            continue

    return out


def build_team_game_table(client: ESPNClient, events: List[Dict[str, Any]], keep_incomplete: bool = False) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for ev in events:
        minimal = parse_scoreboard_event_minimal(ev)
        if not minimal:
            continue

        eid = minimal["event_id"]
        game_date = minimal["game_date"]
        completed = minimal["completed"]

        if (not keep_incomplete) and (not completed):
            continue

        summary = client.get_summary(eid) if completed else None
        team_stats = extract_team_stats_from_summary(summary) if summary else {}

        teams = minimal["teams"]
        if len(teams) != 2:
            continue

        a, b = teams[0], teams[1]

        for tm in teams:
            points = tm["score"]
            opp = b if tm["team_id"] == a["team_id"] else a
            opp_points = opp["score"]

            if completed and (pd.isna(points) or pd.isna(opp_points)):
                continue

            r = {
                "event_id": str(eid),
                "game_date": game_date,
                "team_id": tm["team_id"],
                "team": tm["team"],
                "abbr": tm["abbr"],
                "home_away": tm["home_away"],
                "completed": completed,
                "points": points,
                "opp_points": opp_points,
            }

            stats = team_stats.get(tm["team_id"], {})
            for k, v in stats.items():
                r[f"stat_{k}"] = v

            rows.append(r)

        time.sleep(client.sleep)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df["event_id"] = df["event_id"].astype(str)
    df["team_id"] = df["team_id"].astype(str)

    df = df.loc[:, ~df.columns.duplicated()].copy()

    for c in df.columns:
        if c.startswith("stat_") or c in ("points", "opp_points"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


# -----------------------------
# Dataset building (matchups + rolling)
# -----------------------------
def make_matchup_dataset(team_game: pd.DataFrame, window: int = ROLL_WINDOW) -> Tuple[pd.DataFrame, List[str]]:
    tg = team_game.copy()
    tg = tg.dropna(subset=["event_id", "team_id", "game_date"]).copy()
    tg = tg.sort_values(["team_id", "game_date", "event_id"]).reset_index(drop=True)

    numeric_cols = [c for c in tg.columns if pd.api.types.is_numeric_dtype(tg[c])]
    exclude = {"points", "opp_points"}
    numeric_cols = [c for c in numeric_cols if c not in exclude]

    for c in numeric_cols + ["points", "opp_points"]:
        tg[f"roll_{c}_l{window}"] = (
            tg.groupby("team_id")[c]
              .shift(1)
              .rolling(window)
              .mean()
        )

    base_cols = ["event_id", "game_date", "team_id", "team", "abbr", "home_away", "points"]
    g = tg[base_cols].copy().sort_values(["event_id", "home_away"])

    pairs = []
    for eid, grp in g.groupby("event_id"):
        if len(grp) != 2:
            continue
        home = grp[grp["home_away"] == "home"]
        away = grp[grp["home_away"] == "away"]
        if len(home) != 1 or len(away) != 1:
            continue
        home = home.iloc[0]
        away = away.iloc[0]

        pairs.append({
            "event_id": str(eid),
            "game_date": home["game_date"],
            "home_team": home["team"],
            "away_team": away["team"],
            "home_id": home["team_id"],
            "away_id": away["team_id"],
            "pts_home": home["points"],
            "pts_away": away["points"],
            "y_home_win": int(home["points"] > away["points"])
                if (not pd.isna(home["points"]) and not pd.isna(away["points"]))
                else np.nan,
        })

    games = pd.DataFrame(pairs)
    if games.empty:
        return games, []

    roll_cols = [c for c in tg.columns if c.startswith("roll_")]
    feats = tg[["event_id", "team_id"] + roll_cols].copy()

    df = games.merge(feats, left_on=["event_id", "home_id"], right_on=["event_id", "team_id"], how="left")
    df = df.drop(columns=["team_id"]).rename(columns={c: f"h_{c}" for c in roll_cols})

    df = df.merge(feats, left_on=["event_id", "away_id"], right_on=["event_id", "team_id"], how="left")
    df = df.drop(columns=["team_id"]).rename(columns={c: f"a_{c}" for c in roll_cols})

    feature_cols: List[str] = []
    for c in roll_cols:
        hc, ac = f"h_{c}", f"a_{c}"
        dc = f"d_{c}"
        if hc in df.columns and ac in df.columns:
            df[dc] = df[hc] - df[ac]
            feature_cols.append(dc)

    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df = df.dropna(subset=["game_date"]).sort_values("game_date").reset_index(drop=True)

    return df, feature_cols


# -----------------------------
# Elastic-Net Logistic Training + Calibration
# -----------------------------
def train_calibrated_elastic_net(
    df: pd.DataFrame,
    features: List[str],
) -> Tuple[Any, SimpleImputer, pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    """
    70/15/15 time split:
      train -> fit Elastic-Net LogReg
      cal   -> sigmoid calibration
      test  -> evaluate
    Also does simple grid search over (C, l1_ratio) using cal set logloss.
    """
    df = df.copy()
    df = df.dropna(subset=["y_home_win"]).copy()
    df["y_home_win"] = df["y_home_win"].astype(int)

    for c in features:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    feats = [c for c in features if df[c].notna().any()]
    if DROP_CONSTANT_FEATURES:
        feats = [c for c in feats if df[c].nunique(dropna=True) > 1]

    if MIN_NON_NULL_RATE > 0 and len(feats) > 0:
        coverage = df[feats].notna().mean()
        feats = coverage[coverage >= MIN_NON_NULL_RATE].index.tolist()

    print(f"Features available: {len(features)}")
    print(f"Features used after cleanup/filter: {len(feats)}")
    if len(feats) < 5:
        print("WARNING: very few features survived. ESPN stats might not be extracting well yet.")

    df = df.sort_values("game_date").reset_index(drop=True)
    n = len(df)
    cut_train = int(n * 0.70)
    cut_cal = int(n * 0.85)

    train_df = df.iloc[:cut_train]
    cal_df = df.iloc[cut_train:cut_cal]
    test_df = df.iloc[cut_cal:]

    imp = SimpleImputer(strategy="median")
    X_train = imp.fit_transform(train_df[feats])
    y_train = train_df["y_home_win"].values

    X_cal = imp.transform(cal_df[feats])
    y_cal = cal_df["y_home_win"].values

    X_test = imp.transform(test_df[feats])
    y_test = test_df["y_home_win"].values

    best = None
    best_score = float("inf")

    print("\n--- Elastic-Net grid search (using CAL logloss) ---")
    for C in ELASTIC_C_VALUES:
        for l1r in ELASTIC_L1_RATIOS:
            base = LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                l1_ratio=l1r,
                C=C,
                max_iter=8000,
                random_state=42,
            )
            base.fit(X_train, y_train)

            cal_model = CalibratedClassifierCV(base, method="sigmoid", cv="prefit")
            cal_model.fit(X_cal, y_cal)

            p_cal = cal_model.predict_proba(X_cal)[:, 1]
            score = log_loss(y_cal, p_cal)

            print(f"C={C:<4} l1_ratio={l1r:<4} cal_logloss={score:.4f}")

            if score < best_score:
                best_score = score
                best = (C, l1r)

    bestC, bestL1 = best
    print(f"\nBEST elastic-net params: C={bestC}, l1_ratio={bestL1}, cal_logloss={best_score:.4f}")

    base_best = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=bestL1,
        C=bestC,
        max_iter=8000,
        random_state=42,
    )
    base_best.fit(X_train, y_train)

    cal = CalibratedClassifierCV(base_best, method="sigmoid", cv="prefit")
    cal.fit(X_cal, y_cal)

    p_test = cal.predict_proba(X_test)[:, 1]
    y_hat = (p_test >= 0.5).astype(int)

    print("\n=== TEST RESULTS ===")
    print("Test accuracy:", round(accuracy_score(y_test, y_hat), 4))
    print("Test logloss: ", round(log_loss(y_test, p_test), 4))

    return cal, imp, test_df, y_test, p_test, np.array(feats)


# -----------------------------
# Upcoming prediction helper (Sleeper injuries)
# -----------------------------
def predict_upcoming_week(
    client: ESPNClient,
    team_game: pd.DataFrame,
    model: Any,
    imputer: SimpleImputer,
    window: int,
    features_used: np.ndarray,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    sb = client.get_scoreboard(date_yyyymmdd=None)
    if not sb or "events" not in sb:
        return pd.DataFrame(), pd.DataFrame()

    events = sb["events"]
    mini = []
    teamid_to_abbr = {}

    for ev in events:
        parsed = parse_scoreboard_event_minimal(ev)
        if not parsed:
            continue

        for t in parsed["teams"]:
            if t.get("team_id") and t.get("abbr"):
                teamid_to_abbr[str(t["team_id"])] = str(t["abbr"]).upper()

        if parsed["completed"]:
            continue

        teams = parsed["teams"]
        if len(teams) != 2:
            continue

        home = [t for t in teams if t["home_away"] == "home"]
        away = [t for t in teams if t["home_away"] == "away"]
        if len(home) != 1 or len(away) != 1:
            continue
        home, away = home[0], away[0]

        mini.append({
            "event_id": parsed["event_id"],
            "game_date": parsed["game_date"],
            "home_team": home["team"],
            "away_team": away["team"],
            "home_id": home["team_id"],
            "away_id": away["team_id"],
            "home_abbr": str(home.get("abbr") or teamid_to_abbr.get(str(home["team_id"]), "")).upper(),
            "away_abbr": str(away.get("abbr") or teamid_to_abbr.get(str(away["team_id"]), "")).upper(),
        })

    upcoming = pd.DataFrame(mini)
    if upcoming.empty:
        return upcoming, pd.DataFrame()

    tg = team_game.copy()
    tg = tg.dropna(subset=["team_id", "game_date"]).sort_values(["team_id", "game_date", "event_id"]).reset_index(drop=True)

    numeric_cols = [c for c in tg.columns if pd.api.types.is_numeric_dtype(tg[c])]
    exclude = {"points", "opp_points"}
    numeric_cols = [c for c in numeric_cols if c not in exclude]

    for c in numeric_cols + ["points", "opp_points"]:
        tg[f"roll_{c}_l{window}"] = tg.groupby("team_id")[c].shift(1).rolling(window).mean()

    roll_cols = [c for c in tg.columns if c.startswith("roll_")]
    last = tg.sort_values("game_date").groupby("team_id").tail(1)[["team_id"] + roll_cols].copy()

    df = upcoming.merge(last, left_on="home_id", right_on="team_id", how="left").drop(columns=["team_id"])
    df = df.rename(columns={c: f"h_{c}" for c in roll_cols})

    df = df.merge(last, left_on="away_id", right_on="team_id", how="left").drop(columns=["team_id"])
    df = df.rename(columns={c: f"a_{c}" for c in roll_cols})

    for c in roll_cols:
        df[f"d_{c}"] = df.get(f"h_{c}") - df.get(f"a_{c}")

    for c in features_used:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    X = imputer.transform(df[features_used])
    p_base = model.predict_proba(X)[:, 1]
    df["p_home_win_base"] = p_base

    injuries_long = pd.DataFrame(columns=["team", "player", "pos", "injury_status", "bucket"])

    if USE_INJURY_ADJUSTMENT:
        players_db = load_sleeper_players(force_refresh=False)

        needed_abbr = sorted(set([a for a in df["home_abbr"].tolist() + df["away_abbr"].tolist() if a]))
        inj_feats_by_abbr: Dict[str, Dict[str, float]] = {}
        inj_table_by_abbr: Dict[str, pd.DataFrame] = {}

        for abbr in needed_abbr:
            feats, inj_df = summarize_team_injuries_from_sleeper(players_db, abbr)
            inj_feats_by_abbr[abbr] = feats
            inj_table_by_abbr[abbr] = inj_df

        all_tables = []
        for abbr, inj_df in inj_table_by_abbr.items():
            if inj_df is not None and not inj_df.empty:
                all_tables.append(inj_df.copy())
        if all_tables:
            injuries_long = pd.concat(all_tables, ignore_index=True)

        def _feats(abbr: str) -> Dict[str, float]:
            return inj_feats_by_abbr.get(str(abbr).upper(), {
                "inj_out_total": 0.0, "inj_doubtful_total": 0.0, "inj_questionable_total": 0.0, "inj_ir_total": 0.0,
                "inj_qb_out": 0.0, "inj_qb_doubtful": 0.0, "inj_qb_questionable": 0.0
            })

        h_feats_series = df["home_abbr"].apply(_feats)
        a_feats_series = df["away_abbr"].apply(_feats)

        h_inj = pd.DataFrame(h_feats_series.tolist()).add_prefix("h_")
        a_inj = pd.DataFrame(a_feats_series.tolist()).add_prefix("a_")
        df = pd.concat([df.reset_index(drop=True), h_inj, a_inj], axis=1)

        p_adj = []
        for i in range(len(df)):
            p0 = float(df.loc[i, "p_home_win_base"])

            home_feats = {k.replace("h_", ""): float(df.loc[i, k]) for k in h_inj.columns}
            away_feats = {k.replace("a_", ""): float(df.loc[i, k]) for k in a_inj.columns}

            pen_home = injury_logit_penalty(home_feats)
            pen_away = injury_logit_penalty(away_feats)

            z = logit(p0) - pen_home + pen_away
            p_adj.append(sigmoid(z))

        df["p_home_win"] = np.array(p_adj)
    else:
        df["p_home_win"] = df["p_home_win_base"]

    df["fair_american_home"] = df["p_home_win"].apply(to_american_odds).round(1)
    df["fair_american_away"] = (1 - df["p_home_win"]).apply(to_american_odds).round(1)

    df = df.sort_values("game_date").reset_index(drop=True)
    return df, injuries_long


# -----------------------------
# Main
# -----------------------------
def main():
    client = ESPNClient(sleep=SLEEP_BETWEEN_CALLS)

    print("\n=== 1) Collect historical events ===")
    events_hist = collect_events_by_dates(client, days_back=DEFAULT_N_DAYS_HISTORY)
    print(f"Events collected (raw): {len(events_hist)}")

    print("\n=== 2) Build team-game table (completed games) ===")
    team_game = build_team_game_table(client, events_hist, keep_incomplete=False)
    if team_game.empty:
        print("No completed games found. Exiting.")
        return

    print("team_game shape:", team_game.shape)
    print("team_game date range:", team_game["game_date"].min(), "→", team_game["game_date"].max())

    print("\n=== 3) Build matchup dataset + rolling features ===")
    df, FEATURE_COLS = make_matchup_dataset(team_game, window=ROLL_WINDOW)
    if df.empty or not FEATURE_COLS:
        print("Could not build matchup dataset / no features. Exiting.")
        return

    print("Matchups:", len(df))
    print("Feature candidates:", len(FEATURE_COLS))
    print("Feature example:", FEATURE_COLS[:10])

    print("\n=== 4) Train + calibrate Elastic-Net Logistic model ===")
    model, imputer, test_df, y_test, p_test, FEATURES_USED = train_calibrated_elastic_net(df, FEATURE_COLS)

    out = test_df[["game_date", "home_team", "away_team", "pts_home", "pts_away", "y_home_win"]].copy()
    out["p_home_win"] = np.round(p_test, 3)
    out["fair_american_home"] = out["p_home_win"].apply(to_american_odds).round(1)
    out["fair_american_away"] = (1 - out["p_home_win"]).apply(to_american_odds).round(1)

    latest_day = out["game_date"].max()
    week_start = latest_day - pd.Timedelta(days=7)

    print("\n=== LATEST WEEK (COMPLETED, from TEST SPLIT) ===")
    print(
        out[out["game_date"] >= week_start]
        .sort_values("game_date", ascending=False)
        .head(30)
        .to_string(index=False)
    )

    print("\n=== 5) Predict current-week UPCOMING games (injury-adjusted) ===")
    upcoming, injuries_long = predict_upcoming_week(
        client=client,
        team_game=team_game,
        model=model,
        imputer=imputer,
        window=ROLL_WINDOW,
        features_used=FEATURES_USED,
    )

    if upcoming.empty:
        print("No upcoming games found (or ESPN scoreboard unavailable).")
    else:
        cols = [
            "game_date", "away_team", "home_team",
            "p_home_win_base", "p_home_win",
            "fair_american_home", "fair_american_away",
            "home_abbr", "away_abbr",
            "h_inj_out_total", "a_inj_out_total",
            "h_inj_qb_out", "a_inj_qb_out",
            "h_inj_questionable_total", "a_inj_questionable_total",
        ]
        cols = [c for c in cols if c in upcoming.columns]
        print(upcoming[cols].head(30).to_string(index=False))

        if PRINT_UPCOMING_INJURY_TABLE:
            print("\n=== UPCOMING INJURIES (Sleeper) ===")
            if injuries_long is None or injuries_long.empty:
                print("No injuries listed by Sleeper for teams in the upcoming slate.")
            else:
                print(injuries_long.head(250).to_string(index=False))

    print("\n=== Done ===")
    print("Features used:", len(FEATURES_USED))
    print("First 25:", FEATURES_USED[:25])


main()



=== 1) Collect historical events ===
Events collected (raw): 331

=== 2) Build team-game table (completed games) ===
team_game shape: (662, 33)
team_game date range: 2025-08-01 00:00:00 → 2026-01-18 23:30:00

=== 3) Build matchup dataset + rolling features ===
Matchups: 331
Feature candidates: 27
Feature example: ['d_roll_completed_l5', 'd_roll_stat_firstDowns_l5', 'd_roll_stat_firstDownsPassing_l5', 'd_roll_stat_firstDownsRushing_l5', 'd_roll_stat_firstDownsPenalty_l5', 'd_roll_stat_thirdDownEff_l5', 'd_roll_stat_fourthDownEff_l5', 'd_roll_stat_totalOffensivePlays_l5', 'd_roll_stat_totalYards_l5', 'd_roll_stat_yardsPerPlay_l5']

=== 4) Train + calibrate Elastic-Net Logistic model ===
Features available: 27
Features used after cleanup/filter: 20

--- Elastic-Net grid search (using CAL logloss) ---


/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


C=0.1  l1_ratio=0.05 cal_logloss=0.6340
C=0.1  l1_ratio=0.15 cal_logloss=0.6347


/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


C=0.1  l1_ratio=0.3  cal_logloss=0.6392
C=0.1  l1_ratio=0.5  cal_logloss=0.6433
C=0.1  l1_ratio=0.7  cal_logloss=0.6435


...

C=3.0  l1_ratio=0.5  cal_logloss=0.6329


/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


C=3.0  l1_ratio=0.7  cal_logloss=0.6328

BEST elastic-net params: C=0.7, l1_ratio=0.7, cal_logloss=0.6307


/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(



=== TEST RESULTS ===
Test accuracy: 0.58
Test logloss:  0.6455

=== LATEST WEEK (COMPLETED, from TEST SPLIT) ===
          game_date            home_team            away_team  pts_home  pts_away  y_home_win  p_home_win  fair_american_home  fair_american_away
2026-01-18 23:30:00        Chicago Bears     Los Angeles Rams      17.0      20.0           0       0.569              -132.0               132.0
2026-01-18 20:00:00 New England Patriots       Houston Texans      28.0      16.0           1       0.693              -225.7               225.7
2026-01-18 01:00:00     Seattle Seahawks  San Francisco 49ers      41.0       6.0           1       0.442               126.2              -126.2
2026-01-17 21:30:00       Denver Broncos        Buffalo Bills      33.0      30.0           1       0.504              -101.6               101.6
2026-01-13 01:15:00  Pittsburgh Steelers       Houston Texans       6.0      30.0           0       0.559              -126.8               126.8
2026-01-12